# Load Business Vault

This notebook rebuilds and loads the `business_vault` schema from the already-loaded `raw_vault` schema.

Prerequisites:
- `bronze` schema loaded
- `raw_vault` schema loaded
- PostgreSQL connection configured in `pgsql/db.py`

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
candidates = [cwd, cwd / "pgsql", cwd.parent / "pgsql"]

for candidate in candidates:
    if (candidate / "load_business_vault.py").exists():
        sys.path.insert(0, str(candidate))
        print(f"Using pgsql module path: {candidate}")
        break
else:
    raise FileNotFoundError("Could not find load_business_vault.py")

## Run Loader

This recreates `business_vault` and reloads all BV master and xref tables.

In [ ]:
from load_business_vault import main as load_business_vault

load_business_vault()

## Verify Counts

In [ ]:
import psycopg
from db import connection_details

with psycopg.connect(**connection_details) as con:
    with con.cursor() as cur:
        cur.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'business_vault'
              AND table_type = 'BASE TABLE'
            ORDER BY table_name
        """)
        tables = [row[0] for row in cur.fetchall()]

        for table in tables:
            cur.execute(f"SELECT COUNT(*) FROM business_vault.{table}")
            print(f"{table}: {cur.fetchone()[0]}")

## Verify References

In [ ]:
from verify_vault_integrity import main as verify_vault_integrity

verify_vault_integrity()

## Sample BV Query

Two persons with home policies. This uses `business_vault` for master data and `raw_vault` only as the relationship bridge.

In [ ]:
sample_query = r"""
WITH person_master AS (
    SELECT
        global_person_identifier,
        'NATURAL' AS person_type,
        full_name AS person_name,
        personal_email,
        home_phone
    FROM business_vault.bv_natural_person_master
    UNION ALL
    SELECT
        global_person_identifier,
        'LEGAL' AS person_type,
        organization AS person_name,
        email_address AS personal_email,
        phone_number AS home_phone
    FROM business_vault.bv_legal_person_master
),
home_xref AS (
    SELECT global_home_identifier, rawdv_hashkey AS home_hash_key
    FROM business_vault.bv_home_master_xref
    WHERE rawdv_source_name = 'CRM'
)
SELECT DISTINCT
    p.global_person_identifier,
    p.person_type,
    p.person_name,
    p.personal_email,
    h.global_home_identifier,
    h.policy_identifier,
    h.home_type,
    h.home_location,
    h.global_product_identifier AS product_id
FROM person_master p
JOIN raw_vault.hub_person hp
  ON hp.person_id = split_part(p.global_person_identifier, '||', 1)
 AND hp.record_source = 'CRM'
JOIN raw_vault.link_customer_person lcp
  ON lcp.person_hash_key = hp.person_hash_key
JOIN raw_vault.link_policy_customer lpc
  ON lpc.customer_hash_key = lcp.customer_hash_key
JOIN raw_vault.link_policy_insured_object lpio
  ON lpio.policy_hash_key = lpc.policy_hash_key
JOIN raw_vault.link_insured_object_home lioh
  ON lioh.insured_object_hash_key = lpio.insured_object_hash_key
JOIN home_xref hx
  ON hx.home_hash_key = lioh.home_hash_key
JOIN business_vault.bv_home_master h
  ON h.global_home_identifier = hx.global_home_identifier
ORDER BY p.global_person_identifier, h.policy_identifier
LIMIT 2;
"""

with psycopg.connect(**connection_details) as con:
    with con.cursor() as cur:
        cur.execute(sample_query)
        print([desc.name for desc in cur.description])
        for row in cur.fetchall():
            print(row)